# 04 — Performance and Batching

The previous notebooks computed distances between a handful of points. Real datasets are larger. The US cities dataset used in this notebook has **16,583 entries**. The world dataset has **149,102**.

At that scale, how you write the loop matters. This notebook covers the patterns that keep distance computation fast: list comprehensions over explicit loops, pre-filtering with bounding boxes before calling Haversine, `heapq.nsmallest` for top-N without sorting everything, and a light pass through NumPy vectorization for the cases where even that is not fast enough.

## 1. Load the Data

The world cities file is organized by country code. Pull the US cities into a flat list and inspect the structure.

In [ ]:
import json
import math
import time
import heapq
from pathlib import Path

DATA_PATH = Path("../../../../Resources/Data")

with open(DATA_PATH / ".world_cities_large.json") as f:
    world_cities = json.load(f)

# Pull US cities and normalize coordinates to floats
us_cities = [
    {
        "name": c["city-name"],
        "lon":  float(c["lon"]),
        "lat":  float(c["lat"]),
    }
    for c in world_cities["US"]
]

# All cities as a flat list (for world-scale queries)
all_cities = [
    {
        "name": c["city-name"],
        "country": c["country-code"],
        "lon":  float(c["lon"]),
        "lat":  float(c["lat"]),
    }
    for country_list in world_cities.values()
    for c in country_list
]

print(f"US cities:    {len(us_cities):>7,}")
print(f"World cities: {len(all_cities):>7,}")
print()
print("Sample:", us_cities[0])

## 2. The Naive Approach — Explicit Loop

The most readable version: a `for` loop that builds a list of results one at a time.

In [ ]:
def haversine_km(p1, p2):
    R = 6371.0
    lat1, lon1 = math.radians(p1[1]), math.radians(p1[0])
    lat2, lon2 = math.radians(p2[1]), math.radians(p2[0])
    d_lat = lat2 - lat1
    d_lon = lon2 - lon1
    a = math.sin(d_lat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(d_lon / 2)**2
    return 2 * R * math.asin(math.sqrt(a))


query = (-98.49, 33.91)   # Wichita Falls

# Naive: explicit loop, append to list
def distances_loop(query, cities):
    results = []
    for city in cities:
        d = haversine_km(query, (city["lon"], city["lat"]))
        results.append((d, city["name"]))
    return results

t0 = time.perf_counter()
results_loop = distances_loop(query, us_cities)
t1 = time.perf_counter()

print(f"Loop over {len(us_cities):,} US cities: {(t1-t0)*1000:.1f} ms")
print(f"Result count: {len(results_loop):,}")

## 3. List Comprehension — Same Work, Less Overhead

A list comprehension does the same thing as the loop but avoids the repeated `append` call and the attribute lookup on `results`. The speedup is modest in pure Python, but the readability gain is real and it composes cleanly with `sorted`, `min`, and `filter`.

In [ ]:
# List comprehension version
def distances_comprehension(query, cities):
    return [
        (haversine_km(query, (c["lon"], c["lat"])), c["name"])
        for c in cities
    ]

t0 = time.perf_counter()
results_comp = distances_comprehension(query, us_cities)
t1 = time.perf_counter()

print(f"Comprehension over {len(us_cities):,} US cities: {(t1-t0)*1000:.1f} ms")

# Verify both approaches give the same answer
assert sorted(results_loop) == sorted(results_comp)
print("Results match ✓")

## 4. Sorting at Scale

Once you have a list of `(distance, name)` tuples, you typically want the nearest N. Two options:

- **`sorted(results)[:n]`** — sorts the entire list, then slices. Cost: O(k log k) where k is the full dataset size.
- **`heapq.nsmallest(n, results)`** — maintains a partial sort, only tracking the n smallest. Cost: O(k log n). Substantially faster when n is small relative to k.

For finding the top 5 out of 16,000 records, `heapq.nsmallest` is the right tool.

In [ ]:
N = 5

# Option A: full sort then slice
t0 = time.perf_counter()
top_sorted = sorted(results_comp)[:N]
t1 = time.perf_counter()
sorted_ms = (t1 - t0) * 1000

# Option B: heapq.nsmallest
t0 = time.perf_counter()
top_heap = heapq.nsmallest(N, results_comp)
t1 = time.perf_counter()
heap_ms = (t1 - t0) * 1000

print(f"sorted()[:n]        {sorted_ms:.2f} ms")
print(f"heapq.nsmallest()   {heap_ms:.2f} ms")
print(f"Speedup:            {sorted_ms / heap_ms:.1f}x\n")

print("Top 5 nearest US cities to Wichita Falls:")
for d, name in top_heap:
    print(f"  {name:<30}  {d:.1f} km")

## 5. Bbox Pre-Filter

Haversine is the bottleneck — every call does six trig operations. The trick is to avoid calling it on candidates that are obviously too far.

A bounding box check uses only subtraction and comparison — essentially free. If you pre-filter with a bbox to eliminate candidates outside a rough geographic window, you only call Haversine on the survivors.

For a 200 km radius at ~34°N:
- 200 km ÷ 111.32 km/° ≈ **1.80° latitude margin**
- 200 km ÷ 93 km/° ≈ **2.15° longitude margin** (adjusted for latitude)

In [ ]:
def within_radius_naive(query, cities, radius_km):
    """Check every city with Haversine — no pre-filter."""
    qlon, qlat = query
    return [c for c in cities if haversine_km(query, (c["lon"], c["lat"])) <= radius_km]


def within_radius_bbox(query, cities, radius_km):
    """Pre-filter with bbox, then confirm survivors with Haversine."""
    qlon, qlat = query

    # Degree margins — slightly generous to avoid false exclusions at box edges
    lat_margin = radius_km / 111.32
    lon_margin = radius_km / (111.32 * math.cos(math.radians(qlat)))

    min_lat = qlat - lat_margin
    max_lat = qlat + lat_margin
    min_lon = qlon - lon_margin
    max_lon = qlon + lon_margin

    # Cheap bbox pass
    candidates = [
        c for c in cities
        if min_lat <= c["lat"] <= max_lat and min_lon <= c["lon"] <= max_lon
    ]

    # Precise Haversine pass on survivors only
    return [c for c in candidates if haversine_km(query, (c["lon"], c["lat"])) <= radius_km]


RADIUS = 200   # km

t0 = time.perf_counter()
result_naive = within_radius_naive(query, us_cities, RADIUS)
t1 = time.perf_counter()
naive_ms = (t1 - t0) * 1000

t0 = time.perf_counter()
result_bbox = within_radius_bbox(query, us_cities, RADIUS)
t1 = time.perf_counter()
bbox_ms = (t1 - t0) * 1000

print(f"Naive (Haversine all {len(us_cities):,}):    {naive_ms:.1f} ms  →  {len(result_naive)} cities found")
print(f"Bbox pre-filter:                   {bbox_ms:.1f} ms  →  {len(result_bbox)} cities found")
print(f"Speedup: {naive_ms / bbox_ms:.1f}x")
print()
# Sanity check — same results
naive_names = sorted(c["name"] for c in result_naive)
bbox_names  = sorted(c["name"] for c in result_bbox)
print("Results match:", naive_names == bbox_names)

The bbox pass is nearly free compared to Haversine. The larger the dataset and the smaller the radius relative to the total area, the more dramatic the speedup — because most candidates are eliminated before the trig even runs.

This is the same principle behind spatial indexes like R-trees: do cheap rectangle checks first, call expensive geometry tests only on what survives.

## 6. NumPy Vectorization

When the dataset is large enough that even the optimized Python loop is too slow, NumPy can compute all distances in one vectorized pass — no Python-level loop at all. The math is identical; the execution uses compiled C under the hood.

This is the right move when you need to run distance queries repeatedly at interactive speeds on tens of thousands of points.

In [ ]:
import numpy as np

# Pre-build coordinate arrays once — reuse for every query
lons_np = np.array([c["lon"] for c in all_cities])
lats_np = np.array([c["lat"] for c in all_cities])
names_arr = np.array([c["name"] for c in all_cities])

def haversine_vectorized(query_lon, query_lat, lons, lats):
    """
    Vectorized Haversine: compute distances from one query point to all
    points in the lons/lats arrays simultaneously.
    Returns a NumPy array of distances in km.
    """
    R = 6371.0
    qlat = np.radians(query_lat)
    qlon = np.radians(query_lon)
    lat2 = np.radians(lats)
    lon2 = np.radians(lons)

    d_lat = lat2 - qlat
    d_lon = lon2 - qlon

    a = np.sin(d_lat / 2)**2 + np.cos(qlat) * np.cos(lat2) * np.sin(d_lon / 2)**2
    return 2 * R * np.arcsin(np.sqrt(a))


# Time: Python comprehension vs NumPy on the full world dataset
qlon, qlat = query

t0 = time.perf_counter()
dists_loop = [haversine_km(query, (c["lon"], c["lat"])) for c in all_cities]
t1 = time.perf_counter()
loop_ms = (t1 - t0) * 1000

t0 = time.perf_counter()
dists_np = haversine_vectorized(qlon, qlat, lons_np, lats_np)
t1 = time.perf_counter()
np_ms = (t1 - t0) * 1000

print(f"Python loop  ({len(all_cities):,} world cities): {loop_ms:.1f} ms")
print(f"NumPy vector ({len(all_cities):,} world cities): {np_ms:.1f} ms")
print(f"Speedup: {loop_ms / np_ms:.0f}x")

In [ ]:
# Use the vectorized result: top 10 nearest world cities
top_idx = np.argpartition(dists_np, 10)[:10]       # partial sort — fast
top_idx = top_idx[np.argsort(dists_np[top_idx])]   # sort just the 10

print("10 nearest world cities to Wichita Falls:")
for i in top_idx:
    print(f"  {names_arr[i]:<30}  {dists_np[i]:.1f} km")

## 7. Choosing the Right Approach

| Situation | Best approach |
|-----------|--------------|
| < 1,000 points, one-off query | Plain loop or comprehension — fine |
| ~10k–50k points, repeated queries | Comprehension + bbox pre-filter |
| > 50k points, interactive speed needed | NumPy vectorized Haversine |
| > 1M points, or need spatial joins | Proper spatial index (scipy KDTree, shapely STRtree) — out of scope here |

The numpy arrays (`lons_np`, `lats_np`) are built once and reused. That setup cost amortizes instantly when you run more than a handful of queries.

---

## Exercise A — Nearest City to Each Military Base

The `Military_Bases.geojson` file contains 824 US military bases as polygon features. Compute the centroid of each base's exterior ring, then use `haversine_vectorized` to find the nearest world city to each base. Print the 10 base–city pairs with the shortest distance.

In [ ]:
with open(DATA_PATH / 'Military_Bases.geojson') as f:
    bases_geojson = json.load(f)

def polygon_centroid(ring):
    """Compute the mean [lon, lat] of a coordinate ring."""
    lons = [v[0] for v in ring]
    lats = [v[1] for v in ring]
    return sum(lons) / len(lons), sum(lats) / len(lats)

bases = []
for feature in bases_geojson['features']:
    geom = feature['geometry']
    name = feature['properties'].get('featureName', 'Unknown')
    exterior_ring = geom['coordinates'][0][0]
    clon, clat = polygon_centroid(exterior_ring)
    bases.append({'name': name, 'lon': clon, 'lat': clat})

print(f'Loaded {len(bases)} base centroids')
print('Sample:', bases[0])

nearest_pairs = []
for base in bases:
    dists = haversine_vectorized(base['lon'], base['lat'], lons_np, lats_np)
    idx = int(np.argmin(dists))
    nearest_pairs.append((float(dists[idx]), base['name'], names_arr[idx]))

for dist, base_name, city_name in sorted(nearest_pairs)[:10]:
    print(f'{base_name:<35} {city_name:<25} {dist:6.2f} km')


## Exercise B — Cities Within Range of Any Base

Using `within_radius_bbox`, find all world cities within **50 km** of at least one US military base. How many unique cities qualify? Print the top 10 cities that are closest to any base, along with which base they are nearest to.

In [ ]:
city_hits = {}
for base in bases:
    base_coord = (base['lon'], base['lat'])
    nearby = within_radius_bbox(base_coord, all_cities, 50)
    for city in nearby:
        dist = haversine_km(base_coord, (city['lon'], city['lat']))
        key = (city['name'], city['country'])
        best = city_hits.get(key)
        if best is None or dist < best['distance_km']:
            city_hits[key] = {
                'city': city['name'],
                'country': city['country'],
                'base': base['name'],
                'distance_km': dist,
            }

print('Unique cities within 50 km of any base:', len(city_hits))
for row in sorted(city_hits.values(), key=lambda item: item['distance_km'])[:10]:
    print(f"{row['city']:<25} {row['country']:<4} {row['base']:<35} {row['distance_km']:6.2f} km")


## Exercise C — Time Your Own Query

Pick a launch point from the final project or any coordinate you find interesting. Run all four approaches against the full world cities dataset:

1. Explicit loop
2. List comprehension
3. Comprehension + bbox pre-filter
4. NumPy vectorized

Time each one and print a summary table. Find the top 5 nearest cities using the NumPy result.

In [ ]:
my_query = (-98.49, 33.91)

summary = []

t0 = time.perf_counter()
loop_results = distances_loop(my_query, all_cities)
summary.append(('Explicit loop', (time.perf_counter() - t0) * 1000))

t0 = time.perf_counter()
comp_results = distances_comprehension(my_query, all_cities)
summary.append(('List comprehension', (time.perf_counter() - t0) * 1000))

t0 = time.perf_counter()
bbox_candidates = within_radius_bbox(my_query, all_cities, 500)
bbox_results = [(haversine_km(my_query, (c['lon'], c['lat'])), c['name']) for c in bbox_candidates]
summary.append(('BBox pre-filter + Haversine', (time.perf_counter() - t0) * 1000))

t0 = time.perf_counter()
np_dists = haversine_vectorized(my_query[0], my_query[1], lons_np, lats_np)
summary.append(('NumPy vectorized', (time.perf_counter() - t0) * 1000))

print(f"{'Approach':<30} {'Time (ms)':>12}")
print('-' * 44)
for label, ms in summary:
    print(f'{label:<30} {ms:>10.2f}')

print()
print('Top 5 nearest cities by NumPy:')
for idx in np.argsort(np_dists)[:5]:
    print(f'  {names_arr[idx]:<28} {np_dists[idx]:6.2f} km')


A modest speedup usually means the bbox is not excluding very many candidates. That can happen because the query radius is large enough that the rectangle still covers a dense chunk of the dataset, or because the underlying city data is highly clustered around the query so most records survive the cheap filter and still require Haversine anyway.


## Next

In [00 — What Is Bearing](../07-Bearing/00-What_Is_Bearing.ipynb), we move from distance to direction — given two points, compute the angle from one to the other. Combined with distance, bearing gives you the full geometry of a launch-to-target relationship.